# Fine-tune DARE2D — a step-by-step tutorial

**Fine-tuning** (a.k.a. *transfer learning*) means starting from a model that already works and
nudging it to work on *your* data — instead of training a new model from scratch. You keep what the
pretrained model already learned about cells and divisions, and only re-learn the parts that must
change for your images. In practice you need **far less data**, **less compute**, and you
**overfit less** than training from zero.

A DARE2D model has two parts, and fine-tuning treats them differently:

- the **backbone** — the stack of convolutional layers that turn the raw image into useful features.
  This is the expensive, general part; fine-tuning **keeps it frozen** by default.
- the **head** — the final layers that turn those features into this stage's answer (the division
  centre for segmentation; the axis length + angle for regression). This is the cheap,
  task-specific part; fine-tuning **re-trains it** on your data.

You will point the notebook at a pretrained checkpoint, choose how much of the backbone to keep
frozen, set a gentle learning rate, and run. The result is a new `best.pt` that drops straight into
DARE2D inference.

> **PyTorch only.** Fine-tuning runs on PyTorch, so the starting model must be a `.pt` file. The
> curated demo models ship a `.pt` beside each `.h5`; convert your own `.h5` with
> `dare2d-torch/convert_to_torch.py`.

## The five steps

1. **Pick the model and data** — which pretrained checkpoint to start from, which stage, and which
   image sets to train on and validate against.
2. **Choose what to keep vs. adapt** — how much of the backbone stays frozen.
3. **Set the learning rate & schedule** — how fast, and how carefully, to learn.
4. **Set length, augmentation, early stopping** — how long to train and where the output goes.
5. **Launch**, then **verify** the result loads into inference and **use** it.

Run the cells top to bottom. Each step has one small settings cell you can edit, with notes on
*when* you would change each value. A GPU is strongly recommended.

**What you get out:** a fine-tuned `best.pt` plus a `finetune_config.json` record of the run,
written under `models/<run_name>/…`.

## Setup

Locate the DARE2D repository root (so every path resolves) and the fine-tune engine
`training/torch/finetune.py`, which does the actual training. Nothing to edit here.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

# Resolve the repo root robustly (honour DARE2D_BASE_DIR, else walk up to the folder with setup.py)
# -- same pattern as the prediction/retraining notebooks.
def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in (start, *start.parents):
        if (cand / "setup.py").exists():
            return cand
    return start

base_dir = Path(os.environ.get("DARE2D_BASE_DIR", str(_find_repo_root(Path.cwd()))))
os.chdir(base_dir)                       # finetune.py is launched repo-root-relative, like the widget
finetune_py = base_dir / "training" / "torch" / "finetune.py"
assert finetune_py.exists(), f"finetune.py not found under {base_dir}; run from the DARE2d repo root."
print("Base dir  :", base_dir)
print("Engine    :", finetune_py)

## Step 1 · Pick the model and data

Three choices define *what* you fine-tune and *on what*:

- **`base_model`** — the pretrained checkpoint you start from: any DARE2D `.pt` for the chosen
  stage. The default is the curated demo model for set 8.
- **`stage`** — DARE2D works in two stages, and you fine-tune **one at a time**:
  - `"segmentation"` — the U-Net that finds each division's **centre**.
  - `"regression"` — the CNN that estimates the division **axis** (length + angle) around a centre.
- **`test_set` / `train_sets`** — the demo dataset ships 8 annotated image sets and trains
  **leave-one-out**: the `test_set` is *held out* and used only to measure progress (validation),
  while `train_sets` are the sets the model learns from. Leave `train_sets` blank to use every set
  except the held-out one.

**For your own data:** point `base_model` at your `.pt`, set `stage` to the stage you want to adapt,
and choose which set is held out for validation.

In [ ]:
stage       = "regression"               # "regression" or "segmentation"
experiment  = {"regression": "regression2d", "segmentation": "segmentation2d"}[stage]

# Any DARE2D .pt for that stage. Default = the curated demo checkpoint for set 8.
base_model  = (base_dir / "models" / "demo" / "neuroepithelium"
               / f"{stage}_checkpoints" / "checkpoints_set_8_all_but_target" / "best.pt")

test_set    = 8                          # held-out validation set
train_sets  = ""                         # comma list (e.g. "1,2,3"); blank = all other sets

assert base_model.exists(), (
    f"base model not found: {base_model}\n"
    "Fetch the demo data (napari 'Download DARE2D data'), or point base_model at your own .pt "
    "(convert a .h5 with dare2d-torch/convert_to_torch.py).")
print(f"stage      = {stage}  ({experiment})")
print(f"base_model = {base_model}")
print(f"test_set   = {test_set}   train_sets = {train_sets or '(all but test)'}")

## Step 2 · What to keep, what to adapt (freezing)

**Freezing** a layer means its weights are held fixed — training cannot change them. By default the
whole backbone is frozen and only the head trains, so the model keeps its general feature extractor
and just re-learns the task-specific output. That is what makes fine-tuning data-efficient.

- **`unfreeze_last`** — how many of the *last* backbone blocks to un-freeze (re-open for training).
  `0` trains the head only (safest, needs least data). Increase to `1`, `2`, … when your images
  differ enough that the top of the backbone should adapt too — at the cost of needing more data.
- **`bn_mode`** — a subtlety specific to **BatchNorm** layers. BatchNorm keeps *running statistics*:
  a stored mean and variance of the values flowing through it, used to normalise them. These are
  **not weights**, so freezing the weights does *not* stop them from drifting. `bn_mode` governs
  them:
  - `"frozen"` (default) — keep the stored mean/variance fixed too. A genuine freeze; use when your
    new data looks like the original.
  - `"adapt"` — let BatchNorm **re-estimate** its mean/variance on your data (the weights still stay
    frozen). Use for a **larger, visibly different** dataset (different microscope, brightness, or
    cell type) where the original statistics no longer fit.

In [ ]:
# What to keep vs. adapt
unfreeze_last = 0          # unfreeze the last N backbone blocks (0 = train the head only)
bn_mode       = "frozen"   # frozen-backbone BatchNorm: "frozen" (stats fixed) | "adapt" (re-estimate on new data)


## Step 3 · Learning rate & schedule

The **learning rate (LR)** is how big a step the optimizer takes at each update. Fine-tuning uses a
**small** LR: you are nudging a good model, not rebuilding it, so large steps would erase what it
already knows.

- **`ft_lr`** — the head's learning rate. `1e-4` is a gentle default. Lower it if training is
  unstable or the model "forgets"; raise it slightly if it barely moves.
- **`discriminative` / `backbone_lr_mult`** — if you un-froze part of the backbone (Step 2), it
  should usually learn **slower** than the head, because its features are more general and more
  fragile. With `discriminative = True` the reopened backbone trains at `ft_lr × backbone_lr_mult`
  (e.g. one-tenth of the head's LR).
- **`weight_decay`** — mild AdamW regularisation that keeps weights small and curbs overfitting.
- **`lr_schedule` / `warmup_epochs`** — `"cosine"` first **warms up** the LR over `warmup_epochs`
  (a short ramp that avoids a damaging first jolt), then **anneals** it down a cosine curve for
  smoother convergence. Use `"constant"` to hold the LR fixed instead.
- **`grad_clip`** — caps the gradient size to prevent occasional destabilising updates; `"0"` = off.

Most users only ever touch `ft_lr`; the rest have sensible defaults.

In [ ]:
# Learning rate & schedule
ft_lr            = "1e-4"     # fine-tune LR for the head (small -- you are adapting, not retraining)
discriminative   = True       # give the reopened backbone a lower LR than the head
backbone_lr_mult = "0.1"      # unfrozen-backbone LR = ft_lr * this
weight_decay     = "1e-4"     # AdamW weight decay (mild regularisation)
lr_schedule      = "cosine"   # "cosine" (warmup -> cosine decay) or "constant"
warmup_epochs    = 1          # epochs to ramp the LR up before it decays
grad_clip        = "1.0"      # gradient max-norm; "0" = off


## Step 4 · Length, augmentation, early stopping, output

- **`epochs` / `steps` / `batch_size`** — training length: `steps` optimizer updates per epoch, for
  `epochs` epochs, `batch_size` image crops per update. More epochs = more adaptation, but longer
  runs and more overfitting risk (see `patience`).
- **`augment` / `augment_strength`** — random flips/rotations/intensity changes applied to the
  training crops so the model generalises better; `augment_strength` scales how aggressive they are
  (0–1). Turn augmentation off only for debugging.
- **`patience`** — **early stopping**: if the held-out (validation) loss does not improve for this
  many epochs, stop early and keep the best checkpoint. `0` disables it (train the full `epochs`).
- **`seed`** — fixes the random number generator so a run is reproducible.
- **`run_name`** — names the output folder `models/<run_name>/…`. It **never** overwrites the
  curated demo models, so re-running is safe.

In [ ]:
# How long, augmentation, early stopping, and where output goes
run_name         = "finetune_demo"   # output -> models/<run_name>/... (never overwrites curated demo models)
epochs           = 20
steps            = 500                # optimizer steps per epoch
batch_size       = 32
augment          = True              # apply the training-data augmentations
augment_strength = "1.0"             # scales augmentation probabilities (0..1)
patience         = 0                 # early-stop after N epochs w/o val improvement; 0 = off
seed             = 12345


## Step 5 · Launch the fine-tuning

The next two cells assemble the command for the fine-tune engine and run it. As it trains it streams
progress — `[phase]`, `[step]`, and `[epoch]` markers plus the validation loss — and saves the best
model to `best.pt` along with the `finetune_config.json` record. A GPU is strongly recommended;
**interrupt the kernel** to stop early.

In [ ]:
# Assemble the command line for the fine-tune engine from the settings above.
cmd = [sys.executable, str(finetune_py),
       "--experiment", experiment, "--base-model", str(base_model),
       "--test-set", str(test_set), "--run-name", run_name,
       "--epochs", str(epochs), "--steps", str(steps), "--crop", "256",
       "--batch-size", str(batch_size),
       "--unfreeze-last", str(unfreeze_last), "--bn-mode", bn_mode, "--ft-lr", ft_lr,
       "--backbone-lr-mult", backbone_lr_mult, "--weight-decay", weight_decay,
       "--lr-schedule", lr_schedule, "--warmup-epochs", str(warmup_epochs),
       "--grad-clip", grad_clip, "--augment-strength", augment_strength,
       "--patience", str(patience), "--seed", str(seed)]
if train_sets.strip():
    cmd += ["--train-sets", train_sets.strip()]
if not discriminative:
    cmd += ["--no-discriminative"]
if not augment:
    cmd += ["--no-augment"]


In [ ]:
def run(cmd):
    '''Stream the fine-tune subprocess (same mechanism as the widget's worker) and capture the
    saved checkpoint path from the final '[finetune] DONE ... checkpoint:' line.'''
    print("> " + " ".join(cmd) + "\n")
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            bufsize=1, text=True, encoding="utf-8", errors="replace", env=env)
    ckpt = None
    for line in proc.stdout:
        sys.stdout.write(line)
        if "checkpoint:" in line:
            ckpt = line.split("checkpoint:", 1)[1].strip()
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"fine-tune exited with code {proc.returncode}")
    return ckpt


ckpt_path = run(cmd)
print("\nFine-tuned checkpoint:", ckpt_path)

## Verify: does the result load into inference?

A fine-tune is only useful if the saved model actually loads back for prediction. This loads the new
`best.pt` with the very same loader DARE2D inference uses, and runs one dummy batch through it to
confirm it builds and produces outputs of the right shape. If it prints `OK`, the checkpoint is
ready — no conversion needed.

In [ ]:
import numpy as np
sys.path.insert(0, str(base_dir / "dare2d-torch"))
import torch_backend as tb

if stage == "regression":
    pred = tb.load_torch_regression(ckpt_path)
    length, angle = pred.model.predict(np.random.rand(2, 64, 64, 3).astype("float32"), verbose=0)
    print("OK - fine-tuned regression model loads via the inference backend; "
          f"length {np.asarray(length).shape}, angle {np.asarray(angle).shape}")
else:
    pred = tb.load_torch_segmentation(ckpt_path)
    out = pred.model.predict(np.random.rand(1, 256, 256, 3).astype("float32"), verbose=0)
    print(f"OK - fine-tuned segmentation model loads via the inference backend; output {np.asarray(out).shape}")

## The provenance record (`finetune_config.json`)

Alongside `best.pt`, every fine-tune writes a `finetune_config.json`: the seed, **all** the settings
you chose above, and a fingerprint (name + SHA-256 + size) of the exact base model you started from.
That makes each run reproducible and lets you trace a model back to its parent.

In [ ]:
sidecar = Path(ckpt_path).parent / "finetune_config.json"
print(json.dumps(json.loads(sidecar.read_text(encoding="utf-8")), indent=2))

## Use your fine-tuned model

Point inference at the new checkpoint exactly as you would a curated one:

- **napari plugin** — open **DARE2D division detection** and set the **Regression / Segmentation
  checkpoint** field to the `..._checkpoints` folder holding this `best.pt`.
- **Prediction notebook** — set `reg_dir` / `seg_dir` to that folder.

Because the fine-tuned `best.pt` is byte-compatible with the inference backend (checked in the
verify step above), there is nothing else to do.

## Appendix — correspondence with the napari widget

You do **not** need the plugin to use this notebook. If you *also* use the napari retraining widget, each variable here maps to a widget control and a `finetune.py` flag:

| Notebook variable | napari widget control (fine-tune mode) | `finetune.py` flag |
|---|---|---|
| `base_model` | **Load model to fine-tune (.pt)** | `--base-model` |
| `stage` | **Model** (Regression / Segmentation -- one stage per `.pt`) | `--experiment` |
| `test_set` / `train_sets` | **Test set (held out)** / **Train sets** (leave-one-out) | `--test-set` / `--train-sets` |
| `run_name` | **Run name** | `--run-name` |
| `unfreeze_last` | **Unfreeze last N blocks** | `--unfreeze-last` |
| `bn_mode` | **Frozen-backbone BN** (frozen / adapt) | `--bn-mode` |
| `ft_lr` | **Fine-tune LR** | `--ft-lr` |
| `discriminative` / `backbone_lr_mult` | **Discriminative LR** / **Backbone LR x** | `--no-discriminative` / `--backbone-lr-mult` |
| `weight_decay` | **Weight decay** | `--weight-decay` |
| `lr_schedule` / `warmup_epochs` | **LR schedule** / **Warmup epochs** | `--lr-schedule` / `--warmup-epochs` |
| `grad_clip` | **Grad clip (max-norm)** | `--grad-clip` |
| `augment` / `augment_strength` | **Augmentation** / **Augment strength** | `--no-augment` / `--augment-strength` |
| `patience` | **Early-stop patience** | `--patience` |
| `epochs` / `steps` / `batch_size` | same-named controls | `--epochs` / `--steps` / `--batch-size` |

Everything outside the widget's **Advanced parameters** section runs on the same sensible defaults
you see above.